# Week 2: Learning values and policies

### This notebook will contain

- Discussion on how to find optimal policy/values in the light of model free/based, on-off policies etc.
- In depth discussion on dynamic programming and Bellmann equations, and Monte Carlo

# Reinforcement Learning Method Map

```md
Reinforcement Learning
│
├── Problem setup
│   │
│   ├── MDP
│   │   ├── states: S
│   │   ├── actions: A
│   │   ├── rewards: R
│   │   ├── transition model: p(s', r | s, a)
│   │   └── discount factor: gamma
│   │
│   └── Policy
│       └── pi(a | s)
│
├── Objective
│   │
│   ├── Return
│   │   └── G_t = R_{t+1} + gamma G_{t+1}
│   │
│   └── Maximize expected return
│       └── find pi* that gives large E_pi[G_0]
│
├── Value functions
│   │
│   ├── State-value function
│   │   └── v_pi(s) = expected return from state s
│   │
│   ├── Action-value function
│   │   └── q_pi(s,a) = expected return from action a in state s
│   │
│   └── Optimal values
│       ├── v*(s) = best possible value of state s
│       └── q*(s,a) = best possible value of action a in state s
│
├── Bellman equations
│   │
│   ├── Core idea
│   │   └── value now = immediate reward + discounted value later
│   │
│   ├── Used with known model
│   │   └── exact expectation over p(s', r | s, a)
│   │
│   └── Used with samples
│       └── update from observed transition (s, a, r, s')
│
├── Model-based methods
│   │
│   ├── Assumption
│   │   └── use or learn a model of the environment
│   │
│   ├── Known model
│   │   ├── Dynamic programming
│   │   │   ├── policy evaluation
│   │   │   ├── policy iteration
│   │   │   └── value iteration
│   │   │
│   │   └── Bellman updates use p(s', r | s, a)
│   │
│   └── Learned model
│       ├── Dyna-Q
│       ├── model-based planning
│       └── world models
│
└── Model-free methods
    │
    ├── Assumption
    │   └── do not learn/use explicit p(s', r | s, a)
    │
    ├── Value-based methods
    │   │
    │   ├── Temporal-difference learning
    │   │   ├── SARSA
    │   │   │   └── on-policy
    │   │   │
    │   │   └── Q-learning
    │   │       └── off-policy
    │   │
    │   └── Monte Carlo value learning
    │       └── learns from complete sampled returns
    │
    ├── Policy-based methods
    │   └── learn pi(a | s) directly
    │
    └── Actor-critic methods
        ├── actor learns the policy
        └── critic learns a value function
```

A useful way to read the tree is:

```md
Bellman equations are not a separate algorithm.
They are the recursive principle behind many algorithms.

Dynamic programming uses Bellman equations with a known model.

Q-learning and SARSA use Bellman-style targets from sampled experience.

Q-learning is:
- model-free
- value-based
- temporal-difference
- off-policy

SARSA is:
- model-free
- value-based
- temporal-difference
- on-policy
```


## Model-based and model-free reinforcement learning

A central question in reinforcement learning is whether the agent knows how the environment works. More precisely, does the agent have access to a model of the environment dynamics, or must it learn only from direct interaction? This distinction leads to the two broad categories of **model-based** and **model-free** reinforcement learning.

In a model-based setting, the agent uses information about how actions change the state of the environment. Ideally, it has access to transition probabilities such as $p(s'|s,a)$ and rewards such as $r(s,a)$. This means the agent can plan ahead by asking what is likely to happen after a given action. In a model-free setting, the agent does not use such a model. Instead, it learns from sampled experience: it tries actions, observes rewards and next states, and gradually improves its value estimates or policy.

This distinction is important because different physical problems provide different levels of prior knowledge. In some systems, such as idealized simulations or well-understood equations of motion, a useful model may be available. In other systems, such as noisy quantum devices, complex molecular systems, or robots interacting with the real world, the dynamics may be too uncertain, high-dimensional, or expensive to model exactly. Model-based RL exploits known structure when possible, while model-free RL is useful when learning directly from interaction is more realistic.

## Value functions and the Bellman equation

To make good decisions, the agent needs to estimate how useful different states and actions are. The state-value function of a policy $\pi$ is

$$
v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s].
$$

It tells us the expected future return if the agent starts in state $s$ and then follows policy $\pi$.

Similarly, the action-value function is

$$
q_\pi(s,a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a].
$$

It tells us the expected return from first taking action $a$ in state $s$, and then following policy $\pi$.

The Bellman equation expresses the recursive structure of the value function. Since

$$
G_t = R_{t+1} + \gamma G_{t+1},
$$

the value of a state can be written as immediate reward plus expected future value:

$$
v_\pi(s)
=
\sum_a \pi(a|s)
\sum_{s',r} p(s',r|s,a)
\left[
r + \gamma v_\pi(s')
\right].
$$

This equation is central because it connects present decisions to future consequences. Instead of evaluating entire trajectories from scratch, one can update values locally using information about the next state.

## Dynamic programming

Dynamic programming is a way of solving reinforcement learning problems when the environment model is known. This means that we know the transition probabilities

$$
p(s'|s,a)
$$

and the reward structure. In this case, the Bellman equation can be used directly to compute value functions and improve the policy.

A typical dynamic programming update for policy evaluation is

$$
v_{k+1}(s)
=
\sum_a \pi(a|s)
\sum_{s',r} p(s',r|s,a)
\left[
r + \gamma v_k(s')
\right].
$$

Here, $v_k$ is the current estimate of the value function, and $v_{k+1}$ is the improved estimate after one update. Repeating this update gradually makes the value function more accurate.

Once the value function is known, the policy can be improved by choosing actions that lead to the largest expected return:

$$
\pi_{\text{new}}(s)
=
\arg\max_a
\sum_{s',r} p(s',r|s,a)
\left[
r + \gamma v_\pi(s')
\right].
$$

This gives the dynamic programming principle: an optimal decision can be found by combining the immediate reward with the optimal value of the future state. In other words, a large problem is broken into smaller subproblems.

Dynamic programming is important as a theoretical foundation for reinforcement learning. However, it requires knowledge of the environment model. In many realistic problems, such as robotics, molecular dynamics, or quantum systems, the full model may be unknown, too large, or too expensive to use directly.

## Model-based reinforcement learning

In model-based reinforcement learning, the agent has access to, or tries to learn, a model of the environment. The model describes how the environment responds to actions. In mathematical terms, it tries to represent quantities such as

$$
p(s'|s,a)
$$

and

$$
r(s,a).
$$

If the agent has a good model, it can plan ahead. It can ask: "If I take this action, what is likely to happen next?" This makes model-based RL powerful when the model is accurate.

In physics, this is natural because we often have approximate models of the system, such as equations of motion, Hamiltonians, force fields, or simulators. The agent can use this knowledge to make better decisions with fewer direct trials.

The limitation is that the model may be wrong, incomplete, or too expensive to evaluate. If the system is very complex, uncertain, or noisy, relying too heavily on the model can lead to poor decisions.

## Model-free reinforcement learning

In model-free reinforcement learning, the agent does not need an explicit model of the environment. It does not need to know the transition probabilities $p(s'|s,a)$. Instead, it learns directly from experience by trying actions, observing rewards, and improving its estimates or policy.

This is useful when the environment is difficult to model accurately. For example, a quantum device may be noisy and only partially observable, a molecular system may have a huge configuration space, and a robot may interact with a changing physical environment.

Common model-free methods include:

**Monte Carlo methods:** learn value functions from complete sampled episodes.

**Temporal-difference learning:** updates value estimates step by step, before the episode is finished.

**SARSA:** a temporal-difference method that learns action values while following the current policy.

**Q-learning:** a temporal-difference method that learns an optimal action-value function.

**Policy-gradient methods:** directly optimize the policy parameters.

**Actor-critic methods:** combine a policy model, called the actor, with a value model, called the critic.

## Motivation for the distinction

The difference between model-based and model-free reinforcement learning is mainly about what the agent knows.

In model-based RL, the agent uses knowledge of how the environment works. This allows planning, but requires a useful model.

In model-free RL, the agent learns from interaction without explicitly modeling the environment. This is often more flexible, but may require more data.

For physical systems, both perspectives are useful. If we have reliable equations or simulators, model-based methods can exploit them. If the system is too uncertain, noisy, or complex, model-free methods allow the agent to learn directly from experience.